# Contract protocol · paper companion

In [1]:
from fractions import Fraction as F
from pathlib import Path
import portion as P
import interval_logic as il
from protocol import *


In [2]:
# Clock and negotiation limits
now = F(0)  # Advance between attempts; fixed during each attempt.
PRICES = (F(0), F(10))
ROUND_BUDGET = 3

# Owner: standing commitments, utility targets and access cost
STANDING_BOOKINGS = (Booking(1, P.closedopen(2, 3)),)
C_cmt = standing(*STANDING_BOOKINGS)
OWNER_TARGET = F(4)
OWNER_MINIMA = F(1)
COST_AT_NOW = F(3, 2)
COST_PER_TIME = F(1, 2)

def cost_by_start(start, now):
    return COST_AT_NOW + COST_PER_TIME * (start - now)


# Client: requested service, utility targets and service value
request = Request(2, Service(P.closedopen(1, 5), F(1)), alternatives=())
CLIENT_TARGET = F(6)
CLIENT_MINIMA = F(1)
VALUE_AT_NOW = F(17, 2)
VALUE_DECAY_PER_TIME = F(1, 2)

def value_by_start(start, now):
    return VALUE_AT_NOW - VALUE_DECAY_PER_TIME * (start - now)


# Presentation
EXPORT_PATH = (Path('notebooks') if Path('notebooks').is_dir() else Path('.')) / 'interval-protocol-timelines.html'


## 1. Client and joint task · III-A

For client $k$ and service $\theta=(J,d)$:

$$S_a=\lozenge_J\square_{d)}(acc=k),\qquad
S_u=\lozenge_J\square_{d)}(use=k),\qquad
B=\square(use=k\Rightarrow acc=k).$$
$$C_{\rm client}=\langle S_a\mid B\land S_u\rangle,\qquad
R_{\rm task}=\langle\top\mid B\land S_u\rangle.$$

- `Contract` saturates $G\mapsto G\lor\neg A$ (II-A).
- `grouped` handles disjoint components of $J$ by disjunction; for connected $J$ (as here), it produces one eventuality.
- Access and use have **separate existential witnesses**.


In [3]:
def service_contracts(k, service):
    J, d = service.starts, service.duration
    S_a = grouped(il.Eventually, il.Always(acc.equals(k), d), J)
    S_u = grouped(il.Eventually, il.Always(use.equals(k), d), J)
    B = il.Always(il.Implies(use.equals(k), acc.equals(k)))
    return S_a, S_u, B, Contract(S_a, B & S_u), Contract(True, B & S_u)


In [4]:
S_a, S_u, B, _, _ = service_contracts(request.client, request.service)
table(['Formula', 'Requested service'], [('$S_a$', S_a), ('$S_u$', S_u), ('$B$', B)])

| Formula | Requested service |
| --- | --- |
| $S_a$ | $\lozenge_{[1,5)} \square_{1)} \mathit{acc} = 2$ |
| $S_u$ | $\lozenge_{[1,5)} \square_{1)} \mathit{use} = 2$ |
| $B$ | $\square \left(\mathit{use} = 2 \to \mathit{acc} = 2\right)$ |

## 2. Admission · II-A, III-B

Use the paper's quotient and conjunction. With $G=B\land S_u$ and $\widehat G=\neg S_a\lor G$:

$$(S_a\land G)\lor\neg\widehat G=S_a,$$
$$Q=R_{\rm task}/C_{\rm client}=\langle\widehat G\mid S_a\rangle,\qquad
C_{\rm cmt}^{+}=C_{\rm cmt}\wedge Q=\langle\top\mid L\land S_a\rangle.$$

- Assertions check the computed identities.
- **Overlap:** $[s,s+d)\cap W\ne\varnothing\iff s\in W\oplus(-d,0]$.
  For $W=[a,b)$, forbidden starts are $(a-d,b)$; adjacency is legal.
- Pruning is exact in this model: fixed assignments agree; remaining access is unrestricted.
  Same-client reservations do not block.
- Try the original service, then permitted alternatives **in order**; rebuild each quotient.
  Exhaustion returns `None` (`NO-BLA`).


In [5]:
def admit(request, commitment, now=0):
    now = validate_standing(commitment, now)
    for service in (request.service, *request.alternatives):
        S_a, S_u, B, C_client, R_task = service_contracts(request.client, service)
        Q = R_task.quotient(C_client).normalize(now)
        assert Q.assumption == normalize(~S_a | (B & S_u), now)
        assert Q.guarantee == normalize(S_a, now)
        C_plus = commitment.conjoin(Q)
        assert normalize(C_plus.guarantee, now) == normalize(commitment.guarantee & S_a, now)
        decision = decide_booking(C_plus, now)
        if decision.satisfiable:
            return Admission(request, service, C_client, R_task, Q, decision.normalized,
                             decision.feasible_starts, commitment, now)
    return None


In [6]:
admission = admit(request, C_cmt, now)
assert admission is not None
assert admission.feasible_starts == P.singleton(1) | P.closedopen(3, 5)
table(['Selected service (J,d)', 'Exact feasible starts'], [(f'J={fmt(admission.service.starts)}, d={admission.service.duration}', admission.feasible_starts)])


| Selected service (J,d) | Exact feasible starts |
| --- | --- |
| J=[1,5), d=1 | [1,1] ∪ [3,5) |

## 3. Offers and certificates · II-A, III-C

$$K_s=\langle\top\mid H_s\rangle,\qquad H_s=\square_{[s,s+d)}(acc=k).$$

- **Witness:** same client and duration, $s\in J$, $s\geq now$; hence $H_s\Rightarrow S_a$.
- **Refinement:** $K_s\preceq Q$; conjunction monotonicity and quotient adjunction give

$$C_{\rm cmt}\wedge K_s\preceq C_{\rm cmt}^{+},\qquad
C_{\rm client}\otimes(C_{\rm cmt}\wedge K_s)\preceq C_{\rm cmt}\wedge R_{\rm task}.$$

- **Compatibility, separately:** decide satisfiability of $L\land H_s$.
- **Sampling:** each singleton; otherwise the included left endpoint or midpoint.
  Candidates are alternatives, not commitments.


In [7]:
def certify_offer(admission, booking):
    if not isinstance(booking, Booking):
        raise TypeError('certification requires an unconditional Booking')
    witness = (booking.client == admission.request.client,
               booking.duration == admission.service.duration,
               booking.start in admission.service.starts, booking.start >= admission.now)
    compatible = decide_booking(admission.commitment.conjoin(booking.contract), admission.now).satisfiable
    return Certificate(*witness, compatible)


def generate(admission):
    offers = []
    for s in (() if admission is None else representative_starts(admission.feasible_starts)):
        booking = Booking(admission.request.client, P.closedopen(s, s + admission.service.duration))
        K_s = Contract(True, il.Always(acc.equals(booking.client), booking.interval))
        assert booking.contract == K_s
        certificate = certify_offer(admission, booking)
        assert certificate.certified and certificate.compatible
        offers.append(CertifiedOffer(booking, certificate))
    return tuple(offers)


In [8]:
offers = generate(admission)
table(['Booking', 'Same client', 'Duration', 'Start in J', 'Not past', 'Compatible'],
      [(o.interval, *vars(o.certificate).values()) for o in offers])
assert tuple(o.interval for o in offers) == (P.closedopen(1, 2), P.closedopen(3, 4))


| Booking | Same client | Duration | Start in J | Not past | Compatible |
| --- | --- | --- | --- | --- | --- |
| [1,2) | True | True | True | True | True |
| [3,4) | True | True | True | True | True |

## 4. Bargaining and commitment · III-D

$$u_o(K,q)=q-c(K),\qquad u_c(K,q)=v(K)-q.$$

- **Example:** prices $[0,10]$; targets $(4,6)$; minima $(1,1)$.
- With $\Delta=s-now$: $c(s)=3/2+\Delta/2$, $v(s)=17/2-\Delta/2$.
  Evaluate once at admission time; keep utilities fixed during bargaining.
- **Policies:** earliest legal offer meeting proposer target; acceptance at responder target;
  concession halfway to minimum. `checked_*` validates outputs and target attainability.
- **Rounds:** owner first; one $(K,q)$ per round; rejection concedes both targets and swaps roles.
  Same acceptance policy in the final round.
- Commitments and certified offers stay **fixed throughout the attempt**.


In [9]:
def bargain(offers, policies, targets, minima, prices=(F(0), F(10)), round_budget=3):
    offers = tuple(offers)
    targets, minima, prices = validate_bargaining(offers, targets, minima, prices, round_budget)
    parties, history, proposer = ('owner', 'client'), [], 'owner'
    if not offers:
        return BargainingResult('NO-BLA', None, ())
    initial = {p: checked_proposal(policies, p, offers, targets[p], prices) for p in parties}
    for number in range(1, round_budget + 1):
        responder = 'client' if proposer == 'owner' else 'owner'
        pair = initial['owner'] if number == 1 else checked_proposal(policies, proposer, offers, targets[proposer], prices)
        utilities = policy_utilities(policies, pair)
        accepted = checked_acceptance(policies, responder, pair, targets[responder], minima[responder])
        history.append(BargainRound(number, proposer, tuple(targets.items()), pair, utilities, accepted))
        if accepted:
            return BargainingResult('ACCEPTED', pair, tuple(history))
        if number < round_budget:
            targets = {p: checked_concession(policies, p, targets[p], minima[p]) for p in parties}
            proposer = responder
    return BargainingResult('NO-BLA', None, tuple(history))


In [10]:
policies = affine_policies({o: cost_by_start(o.start, admission.now) for o in offers},
                           {o: value_by_start(o.start, admission.now) for o in offers})
negotiation = bargain(offers, policies,
                      targets={'owner': OWNER_TARGET, 'client': CLIENT_TARGET},
                      minima={'owner': OWNER_MINIMA, 'client': CLIENT_MINIMA},
                      prices=PRICES, round_budget=ROUND_BUDGET)
table(['Round', 'Proposer', 'Targets', 'Booking', 'Price', 'Utilities', 'Accepted'],
      [(r.number, r.proposer, r.targets, r.proposal.offer.interval, r.proposal.price, r.utilities, r.accepted)
       for r in negotiation.history])
assert negotiation.agreement == PricedOffer(offers[0], F(9, 2))


| Round | Proposer | Targets | Booking | Price | Utilities | Accepted |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | owner | (('owner', Fraction(4, 1)), ('client', Fraction(6, 1))) | [1,2) | 6 | (('owner', Fraction(4, 1)), ('client', Fraction(2, 1))) | False |
| 2 | client | (('owner', Fraction(5, 2)), ('client', Fraction(7, 2))) | [1,2) | 9/2 | (('owner', Fraction(5, 2)), ('client', Fraction(7, 2))) | True |

**Acceptance (III-D):** record the exact pair $(K^*,q^*)$, then

$$C_{\rm cmt}\leftarrow C_{\rm cmt}\wedge K^*.$$

- Validate the frozen snapshot and accepted pair; price is ledger metadata.
- `NO-BLA` preserves commitments and ledger.


In [11]:
def commit(admission, offers, negotiation, commitment, ledger=()):
    ledger = tuple(ledger)
    if negotiation.status == 'NO-BLA' and negotiation.agreement is None:
        return CommitResult('NO-BLA', commitment, ledger, None)
    validate_commit(admission, offers, negotiation, commitment, generate(admission))
    K, q = negotiation.agreement.offer.contract, negotiation.agreement.price
    record = BLA(K, q)
    return CommitResult('COMMITTED', commitment.conjoin(K), (*ledger, record), record)


In [12]:
committed = commit(admission, offers, negotiation, C_cmt)
assert committed.agreement == BLA(offers[0].contract, F(9, 2))
table(['Status', 'Accepted booking', 'Price', 'Prior bookings preserved'],
      [(committed.status, negotiation.agreement.offer.interval, committed.agreement.price, len(STANDING_BOOKINGS))])


| Status | Accepted booking | Price | Prior bookings preserved |
| --- | --- | --- | --- |
| COMMITTED | [1,2) | 9/2 | 1 |

## 5. Results and evidence

- **Booking figure:** computed starts, alternative offers and accepted commitment.
- **Signal figure:** illustrative complete behaviors; labels use temporal semantics.
- Later use needs illustrative extra access, outside the BLA; absence leaves access reserved
  but the client task unfulfilled (III-A, III-C).


In [13]:
horizon = protocol_horizon(STANDING_BOOKINGS, request, offers)
booking_figure = plot_booking_progression(STANDING_BOOKINGS, request, admission, offers, negotiation, committed, horizon=horizon)
signal_scenarios = illustrative_signal_scenarios(STANDING_BOOKINGS, negotiation.agreement.offer.booking, offers[1].booking)
signal_figure = plot_signal_traces(signal_scenarios, committed.commitment, admission.client_contract, horizon=horizon)
display_timeline(booking_figure)
display_timeline(signal_figure)


Booking progression

Access and use signals

**Implementation checks** — separate from the mathematical argument:

- Independent feasibility: partition at critical boundaries; test endpoints/interiors using
  interval intersections, without production pruning or decision helpers.
- Complete accepted trace: evaluate owner commitments, client assumption, client guarantee and joint task.


In [14]:
assert independent_feasible_starts(admission.service, STANDING_BOOKINGS, request.client, now) == admission.feasible_starts
trace = joint_witness((*STANDING_BOOKINGS, negotiation.agreement.offer.booking), (*STANDING_BOOKINGS, negotiation.agreement.offer.booking))
checks = check_joint_behavior(trace, committed.commitment, admission.client_contract, admission.task, now)
assert all(checks.values())
table(['Accepted joint trace check', 'Satisfied'], checks.items())
scenario_checks = {name: check_joint_behavior(s['trace'], committed.commitment, admission.client_contract, admission.task, now)
                   for name, s in signal_scenarios.items()}
assert all(c['owner'] for c in scenario_checks.values())
assert [c['joint_task'] for c in scenario_checks.values()] == [True, True, False]
table(['Illustrative behavior', 'Owner', 'Client assumption', 'Client guarantee', 'Joint task'],
      [(name, *c.values()) for name, c in scenario_checks.items()])


| Accepted joint trace check | Satisfied |
| --- | --- |
| owner | True |
| client_assumption | True |
| client_guarantee | True |
| joint_task | True |

| Illustrative behavior | Owner | Client assumption | Client guarantee | Joint task |
| --- | --- | --- | --- | --- |
| Use during the booking | True | True | True | True |
| Later use with additional access | True | True | True | True |
| Client absent | True | True | False | False |

**Boundary/failure cases · III-B–D**

- $[2,3)$ witnesses $S_a$ but conflicts with standing access: certification ≠ compatibility.
- Circular offer $\langle B\mid H\rangle$: no access + client use makes both conditional
  guarantees hold while the task fails. Our offers are unconditional.


In [15]:
collision = certify_offer(admission, Booking(2, P.closedopen(2, 3)))
assert collision.certified and not collision.compatible
cases = [('Adjacency', Service(P.singleton(1), 1), (), P.singleton(1)),
         ('Exact fit', Service(P.closed(1, 2), 1), (), P.singleton(1)),
         ('Blocked', Service(P.singleton(2), 1), (), P.empty()),
         ('Permitted adaptation', Service(P.singleton(2), 1), (Service(P.singleton(3), 1),), P.singleton(3))]
rows = []
for label, service, alternatives, expected in cases:
    result = admit(Request(2, service, alternatives), C_cmt)
    actual = result.feasible_starts if result else P.empty()
    assert actual == expected
    rows.append((label, fmt(actual), 'PASS'))
S_a, S_u, B, C_client, R_task = service_contracts(request.client, admission.service)
counterexample = joint_witness((), (Booking(2, P.closedopen(1, 2)),))
circular = Contract(B, offers[0].contract.guarantee)
assert il.satisfies(C_client.guarantee & circular.guarantee, counterexample, now)
assert not il.satisfies(R_task.guarantee, counterexample, now)
table(['Case', 'Result', 'Check'], [*rows, ('Circular conditional offer', 'Both contracts hold; task fails', 'PASS')])
table(['Finite algebra regression', 'Result'], check_boolean_algebra().items())


| Case | Result | Check |
| --- | --- | --- |
| Adjacency | [1,1] | PASS |
| Exact fit | [1,1] | PASS |
| Blocked | ∅ | PASS |
| Permitted adaptation | [3,3] | PASS |
| Circular conditional offer | Both contracts hold; task fails | PASS |

| Finite algebra regression | Result |
| --- | --- |
| saturated_contracts | 9 |
| triples | 729 |
| passed | True |

- **Algebra regression (II-A):** 9 saturated contracts, 729 triples; operations, quotient adjunction,
  conjunction's greatest-lower-bound property.
- Finite checks concern this complete-behavior model; they establish neither general SAT nor controller synthesis.
- **Offline export:** the same two figure objects, with Plotly embedded; supplementary to this notebook.


In [16]:
export_timelines(EXPORT_PATH, booking_figure, signal_figure)
print(f'Generated {EXPORT_PATH}')


Generated notebooks/interval-protocol-timelines.html
